In [ ]:
'''
Instantiation Pipeline:
1) Full-body validated corpus
2) clean text function
3) chunking function !
4) embedding

5) add func for people to add their own docs
Query Pipeline:
1.1) pre-filtering deterministic lookup bypass
1.2) misc pre-filtering
2) Query embedding (choose: Exact query, Hypothetical Document Embedding, Multi-Query Union)
3) Top-ksearch
3.1) re-ranking top-k
3.2) similarity lower bound threshold refusal trigger
4) chunk token embedding
4.1) LLM-RAG system prompt
'''

'\nInstantiation Pipeline:\n1) Full-body validated corpus\n2) clean text function\n3) chunking function\n4) embedding\n\nQuery Pipeline:\n1.1) pre-filtering deterministic lookup bypass\n1.2) misc pre-filtering\n2) Query embedding (choose: Exact query, Hypothetical Document Embedding, Multi-Query Union)\n3) Top-ksearch\n3.1) re-ranking top-k\n3.2) similarity lower bound threshold refusal trigger\n4) chunk token embedding\n4.1) LLM-RAG system prompt\n'

In [ ]:
#Instantiation Pipeline:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
import bisect


def semantic_chunker(document_list):
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    text_splitter = SemanticChunker(
        embeddings,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=70.0)

    all_chunks = []

    def page_of(offset, page_starts):
        return bisect.bisect_right(page_starts, offset)  # 1-indexed page

    for doc in document_list:
        chunks = text_splitter.create_documents(
            [doc["text"]],
            metadatas=[{"filename": doc["filename"]}])
        cursor = 0
        for chunk_index, chunk in enumerate(chunks):
            start = doc["text"].find(chunk.page_content, cursor)
            cursor = start + len(chunk.page_content)
            chunk.metadata["chunk_index"] = chunk_index
            chunk.metadata["page"] = page_of(start, doc["page_starts"])
        all_chunks.extend(chunks)

    return all_chunks


def chunk_embedder(all_chunks, persist_dir="./index"):
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = Chroma.from_documents(
        all_chunks,
        embeddings,
        persist_directory=persist_dir)   # written to disk so you don't re-embed
    return vectorstore

